In [ ]:
# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

# Change the current directory to the project folder on Google Drive
# This ensures that all file operations happen within this directory
%cd /content/drive/MyDrive/Techlabs_MueSi_2/Colabs/

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1NE_UQElHUZ6Qtv3KBYqjox6rQTbCF7Tk/Techlabs_MueSi_2/Colabs


In [ ]:
import geopandas as gpd

In [ ]:
#!pip install osmnx
#import osmnx as ox
place = "Münster, North Rhine-Westphalia, Germany"

In [ ]:
# Alle Wege in Münster
streets = gpd.read_file("muenster_walk_network.geojson")

In [ ]:
# Lampen
lamps = gpd.read_file("lamps.geojson")

In [ ]:
# Bus-Stops
bus_stops = gpd.read_file("bus_stop.geojson")

In [ ]:
# Bus-Routen
bus_roads = gpd.read_file("bus_roads.geojson")

In [ ]:
# Lärm
noise = gpd.read_file("laerm_straße_nacht.geojson")

# Lärm Polygone vereinfachen, damit spacial join schneller geht
noise = noise.to_crs(epsg=25832)
noise["geometry"] = noise.geometry.simplify(tolerance=40, preserve_topology=True)

In [ ]:
# Unfälle
accidents = gpd.read_file("unfaelle-muenster.geojson")

In [ ]:
#Polizei
police = gpd.read_file("police.geojson")

In [ ]:
#Taxistand
taxi = gpd.read_file("taxi.geojson")

In [ ]:
# Projektion normalisieren
streets = streets.to_crs(epsg=25832)
lamps = lamps.to_crs(epsg=25832)
bus_stops = bus_stops.to_crs(epsg=25832)
bus_roads = bus_roads.to_crs(epsg=25832)
accidents = accidents.to_crs(epsg=25832)
noise = noise.to_crs(epsg=25832)
police = police.to_crs(epsg=25832)
taxi = taxi.to_crs(epsg=25832)

In [ ]:
# Buffer einbauen
lamps["geometry"] = lamps.geometry.buffer(20)
bus_stops["geometry"] = bus_stops.geometry.buffer(30)
bus_roads["geometry"] = bus_roads.geometry.buffer(20)
accidents["geometry"] = accidents.geometry.buffer(1)
police["geometry"] = police.geometry.buffer(40)
taxi["geometry"] = taxi.geometry.buffer(30)

In [ ]:
# Spatial Joins -> bessere Performance?
lamp_join = streets.sjoin(
    lamps[["geometry"]],
    how="inner",
    predicate="intersects"
)

streets["has_lamp"] = streets.index.isin(lamp_join.index)

bus_stop_join = streets.sjoin(
    bus_stops[["geometry"]],
    how="inner",
    predicate="intersects"
)

streets["has_bus_stop"] = streets.index.isin(bus_stop_join.index)

bus_join = streets.sjoin(
    bus_roads[["geometry"]],
    how="inner",
    predicate="intersects"
)

streets["has_bus"] = streets.index.isin(bus_join.index)

accidents_join = streets.sjoin(
    accidents[["geometry"]],
    how="inner",
    predicate="intersects"
)

streets["has_accident"] = streets.index.isin(accidents_join.index)

police_join = streets.sjoin(
    police[["geometry"]],
    how="inner",
    predicate="intersects"
)

streets["has_police"] = streets.index.isin(police_join.index)

taxi_join = streets.sjoin(
    taxi[["geometry"]],
    how="inner",
    predicate="intersects"
)

streets["has_taxi"] = streets.index.isin(taxi_join.index)

noise_join = streets.sjoin(
    noise[["geometry"]],
    how="inner",
    predicate="intersects"
)

streets["has_noise"] = streets.index.isin(noise_join.index)

In [ ]:
# Wohlfühl-Score hinzufügen
streets["comfort_score"] = (
    1.5*streets["has_lamp"].astype(int) +
    streets["has_bus"].astype(int) +
    streets["has_bus_stop"].astype(int) -
    0.5*streets["has_accident"].astype(int) +
    1.5*streets["has_police"].astype(int) +
    streets["has_taxi"].astype(int) -
    0.5*streets["has_noise"].astype(int)
)

In [ ]:
# Hinzufügen einer Spalte, die anzeigt, ob mindestens ein Faktor vorhanden ist
streets["has_any_factor"] = (
    streets["has_lamp"] |
    streets["has_bus"] |
    streets["has_bus_stop"] |
    streets["has_accident"] |
    streets["has_police"] |
    streets["has_taxi"] |
    streets["has_noise"]
)

In [ ]:
# Optimierungen für erhöhte Performance -> Streets-Datensatz minimieren + simplifizieren
streets["geometry"] = streets.geometry.simplify(
    tolerance=15,   # Meter
    preserve_topology=True
)

streets = streets[
    ["geometry", "has_lamp", "has_bus_stop", "has_bus", "has_accident", "has_police", "has_taxi", "has_noise", "comfort_score", "has_any_factor"]
]


In [ ]:
# Visualisierung
# ToDo: LayerControl + Toggle, Night-Mode-Karte, Vergrößerung der Hover-Objekte
# ToDo: neue Daten hinzufügen: Bäume (natural=tree), Radwege, Beleuchtete Geschäfte abends, Unsichere Orte (z. B. Unterführungen)

#Interaktivität-Text für die Popups generieren
def factor_text(row):
    factors = []
    if row["has_lamp"]:
        factors.append("💡 Straßenbeleuchtung")
    if row["has_bus"]:
        factors.append("🚌 Buslinie")
    if row["has_bus_stop"]:
        factors.append("🚌 Haltestelle")
    if row["has_accident"]:
        factors.append("⛔️ Unfälle")
    if row["has_police"]:
        factors.append("👮‍♂️ Polizei")
    if row["has_taxi"]:
        factors.append("🚕 Taxi")
    if row["has_noise"]:
        factors.append("📣 Lärm")
    if not factors:
        return "Keine Wohlfühlfaktoren"
    return ", ".join(factors)

streets["factors"] = streets.apply(factor_text, axis=1)

In [ ]:
# Popup-Generierung

streets["popup"] = (
    "<b>Wohlfühl-Score:</b> " + streets["comfort_score"].astype(str) +
    "<br><b>Faktoren:</b> " + streets["factors"]
)

In [ ]:
# Optional: Export fuer die Weiterverarbeitung in VS Code (Boris)
import os
out_path = "streets.geojson"

# Ensure WGS84 for web maps
streets_out = streets.to_crs(epsg=4326).copy()

# Keep only what we need (adjust if you want more fields)
keep_cols = [c for c in streets_out.columns if c.startswith("has_")] + ["geometry"]
streets_out = streets_out[keep_cols]

streets_out.to_file(out_path, driver="GeoJSON")
print("Wrote:", os.path.abspath(out_path))

Wrote: /content/drive/.shortcut-targets-by-id/1NE_UQElHUZ6Qtv3KBYqjox6rQTbCF7Tk/Techlabs_MueSi_2/Colabs/streets.geojson
